# 01 - WLASL Data Exploration

This notebook loads the WLASL annotations, explores the dataset structure,
and visualizes class distributions, split statistics, video durations, and
sample frames with MediaPipe skeleton overlays.

**Prerequisites:**
- Run `python scripts/download_wlasl.py` to download annotations.
- Optionally have some video files in `data/raw/` for frame visualization.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Annotations

We use the preprocessing module to download and parse the WLASL annotations.
The parser returns a structured DataFrame with video metadata.

In [ ]:
from src.data.preprocess import download_wlasl_annotations, parse_wlasl_annotations

data_dir = PROJECT_ROOT / "data"
annotation_path = download_wlasl_annotations(data_dir / "annotations")

# Parse for WLASL-100 (the recommended starting variant)
df = parse_wlasl_annotations(annotation_path, subset="WLASL100")
print(f"Total samples: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

## 2. Class Distribution

How many video samples does each gloss (word) have? Is the dataset balanced?

In [ ]:
class_counts = df["gloss"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of top-20 classes
class_counts.head(20).plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Top 20 Glosses by Sample Count")
axes[0].set_ylabel("Number of Videos")
axes[0].tick_params(axis="x", rotation=45)

# Histogram of class sizes
axes[1].hist(class_counts.values, bins=30, color="steelblue", edgecolor="white")
axes[1].set_title("Distribution of Class Sizes")
axes[1].set_xlabel("Samples per Class")
axes[1].set_ylabel("Number of Classes")
axes[1].axvline(class_counts.mean(), color="red", linestyle="--", label=f"Mean: {class_counts.mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Min samples per class: {class_counts.min()}")
print(f"Max samples per class: {class_counts.max()}")
print(f"Mean samples per class: {class_counts.mean():.1f}")
print(f"Median samples per class: {class_counts.median():.1f}")

## 3. Split Statistics

The WLASL dataset comes with official train/val/test splits.
Let us verify that all three splits are present and check their sizes.

In [ ]:
split_counts = df["split"].value_counts()
print("Split distribution:")
print(split_counts)
print()

# Per-split class coverage
for split in ["train", "val", "test"]:
    split_df = df[df["split"] == split]
    n_classes = split_df["label_idx"].nunique()
    print(f"{split}: {len(split_df)} samples, {n_classes} classes")

fig, ax = plt.subplots(figsize=(6, 4))
split_counts.plot(kind="bar", ax=ax, color=["steelblue", "coral", "seagreen"])
ax.set_title("Samples per Split")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 4. Signer Distribution

Understanding who the signers are helps assess diversity and potential
signer-specific bias.

In [ ]:
signer_counts = df["signer_id"].value_counts()
print(f"Number of unique signers: {len(signer_counts)}")
print(f"Top 10 signers by video count:")
print(signer_counts.head(10))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(signer_counts)), signer_counts.values, color="steelblue")
ax.set_title("Videos per Signer")
ax.set_xlabel("Signer Index (sorted)")
ax.set_ylabel("Video Count")
plt.tight_layout()
plt.show()

## 5. Sample Frame Visualization

If video files are available in `data/raw/`, we display sample frames
from a few different glosses with MediaPipe skeleton overlay.

In [ ]:
import cv2

raw_dir = data_dir / "raw"
sample_glosses = df["gloss"].unique()[:4]

fig, axes = plt.subplots(len(sample_glosses), 4, figsize=(16, 4 * len(sample_glosses)))
if len(sample_glosses) == 1:
    axes = axes[np.newaxis, :]

for row, gloss in enumerate(sample_glosses):
    gloss_df = df[df["gloss"] == gloss].head(1)
    vid = gloss_df.iloc[0]["video_id"]
    
    # Try to find the video file
    video_path = None
    for ext in [".mp4", ".avi", ".mov"]:
        candidate = raw_dir / f"{vid}{ext}"
        if candidate.exists():
            video_path = candidate
            break
    
    if video_path is not None:
        cap = cv2.VideoCapture(str(video_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        sample_indices = np.linspace(0, max(total_frames - 1, 0), 4, dtype=int)
        
        for col, frame_idx in enumerate(sample_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                axes[row, col].imshow(frame_rgb)
            axes[row, col].set_title(f"{gloss} - frame {frame_idx}")
            axes[row, col].axis("off")
        cap.release()
    else:
        for col in range(4):
            axes[row, col].text(0.5, 0.5, f"Video not found:\n{vid}",
                               ha="center", va="center", fontsize=10)
            axes[row, col].set_title(gloss)
            axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 6. MediaPipe Skeleton on a Sample Frame

Demonstrate what the MediaPipe Holistic output looks like on a single frame.
This shows the pose, hand, and face landmarks that form the input features
for Approach A.

In [ ]:
try:
    import mediapipe as mp
    mp_drawing = mp.solutions.drawing_utils
    mp_holistic = mp.solutions.holistic

    # Find any available video
    sample_vid = None
    for _, row in df.iterrows():
        for ext in [".mp4", ".avi", ".mov"]:
            candidate = raw_dir / f"{row['video_id']}{ext}"
            if candidate.exists():
                sample_vid = candidate
                break
        if sample_vid:
            break

    if sample_vid:
        cap = cv2.VideoCapture(str(sample_vid))
        # Read a frame from the middle of the video
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
        ret, frame = cap.read()
        cap.release()

        if ret:
            with mp_holistic.Holistic(static_image_mode=True) as holistic:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(frame_rgb)

                annotated = frame_rgb.copy()
                if results.pose_landmarks:
                    mp_drawing.draw_landmarks(
                        annotated, results.pose_landmarks,
                        mp_holistic.POSE_CONNECTIONS)
                if results.left_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        annotated, results.left_hand_landmarks,
                        mp_holistic.HAND_CONNECTIONS)
                if results.right_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        annotated, results.right_hand_landmarks,
                        mp_holistic.HAND_CONNECTIONS)

                fig, axes = plt.subplots(1, 2, figsize=(14, 6))
                axes[0].imshow(frame_rgb)
                axes[0].set_title("Original Frame")
                axes[0].axis("off")
                axes[1].imshow(annotated)
                axes[1].set_title("MediaPipe Holistic Landmarks")
                axes[1].axis("off")
                plt.tight_layout()
                plt.show()
    else:
        print("No video files found in data/raw/. Download videos first.")

except ImportError:
    print("MediaPipe not installed. Run: pip install mediapipe")